In [1]:
import pandas as pd
import fasttext
from sklearn.model_selection import train_test_split
import os
import re
import time

# --- CONFIGURAZIONE ---
# Assicurati che il file scaricato si chiami così e sia nella cartella!
VECTORS_FILE = 'model/cc.it.300.vec' 

# 1. PREPARAZIONE DATI (Come prima)
df = pd.read_csv('LLM_DF.csv')

def map_sentiment(star):
    if star >= 4: return 'positive'
    elif star <= 2: return 'negative'
    else: return 'neutral'

df['sentiment'] = df['stars'].apply(map_sentiment)

# Filtriamo via i neutri per la massima precisione
df_binary = df[df['sentiment'] != 'neutral'].copy()

# Bilanciamento
df_pos = df_binary[df_binary['sentiment'] == 'positive']
df_neg = df_binary[df_binary['sentiment'] == 'negative']
min_count = min(len(df_pos), len(df_neg))

df_final = pd.concat([
    df_pos.sample(n=min_count, random_state=42),
    df_neg.sample(n=min_count, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

# Pulizia
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = text.replace('\n', ' ')
    return text

df_final['clean_comment'] = df_final['comment'].apply(clean_text)
df_final['fasttext_line'] = '__label__' + df_final['sentiment'] + ' ' + df_final['clean_comment']

# Split
train, test = train_test_split(df_final, test_size=0.2, random_state=42)
train['fasttext_line'].to_csv('train_bin.txt', index=False, header=False)
test['fasttext_line'].to_csv('test_bin.txt', index=False, header=False)

# --- MODELLO 1: SCRATCH (Quello leggero che abbiamo già fatto) ---
print("🚀 Addestramento Modello 1 (SCRATCH)...")
start_time = time.time()
model_scratch = fasttext.train_supervised(
    input="train_bin.txt", 
    epoch=100, 
    lr=1.0, 
    wordNgrams=2,
    dim=100  # Dimensione standard leggera
)
time_scratch = time.time() - start_time
res_scratch = model_scratch.test("test_bin.txt")

# --- MODELLO 2: PRE-TRAINED (Facebook vectors) ---
if os.path.exists(VECTORS_FILE):
    print(f"🚀 Addestramento Modello 2 (PRE-TRAINED ITALIANO)...")
    print("⚠️  Attenzione: Caricamento vettori da 4GB in corso (può richiedere 2-3 min)...")
    
    start_time = time.time()
    # NOTA: Quando si usano vettori pre-trained, 'dim' deve essere 300 obbligatoriamente
    model_pretrained = fasttext.train_supervised(
        input="train_bin.txt", 
        epoch=25, 
        lr=1.0, 
        wordNgrams=2,
        dim=300,  # DEVE essere 300 per combaciare col file cc.it.300.vec
        pretrainedVectors=VECTORS_FILE
    )
    time_pretrained = time.time() - start_time
    res_pretrained = model_pretrained.test("test_bin.txt")
    
    has_pretrained = True
else:
    print(f"❌ File {VECTORS_FILE} non trovato. Salto il modello pre-trained.")
    has_pretrained = False

# --- TABELLA DI CONFRONTO FINALE ---
print("\n" + "="*60)
print(f"{'METRICA':<20} | {'SCRATCH':<15} | {'PRE-TRAINED (FB)':<15}")
print("="*60)
print(f"{'Precision':<20} | {res_scratch[1]:.4f}          | {res_pretrained[1] if has_pretrained else 'N/A':.4f}")
print(f"{'Recall':<20} | {res_scratch[2]:.4f}          | {res_pretrained[2] if has_pretrained else 'N/A':.4f}")
print(f"{'Tempo Training':<20} | {time_scratch:.2f} sec       | {time_pretrained if has_pretrained else 'N/A':.2f} sec")
print("="*60)

# --- TEST LIVE CONFRONTO (VERSIONE MULTI-FRASE) ---
if has_pretrained:
    print("\n" + "="*80)
    print(f"{'FRASE DI TEST':<50} | {'SCRATCH':<12} | {'PRE-TRAINED'}")
    print("="*80)

    # Definiamo la lista di frasi da testare
    frasi_test = [
        "Il personale era abbastanza freddo ma il cibo buono",
        "Un'esperienza indimenticabile, torneremo sicuramente!",
        "Pessimo servizio, ho aspettato un'ora per un piatto freddo.",
        "L'atmosfera è suggestiva e l'arredamento ricercato.", # Parole ricercate
        "Locale mediocre, non lo consiglierei ai miei amici.",
        "Cibo sublime, ma il conto era davvero esorbitante.",    # Sentimento misto
        "Si mangia bene."                                      # Frase cortissima
    ]

    for frase in frasi_test:
        # 1. Pulizia della frase
        frase_clean = clean_text(frase)
        
        # 2. Predizione con Modello Scratch
        # predict() restituisce una tupla (('__label__positive',), array([0.99]))
        pred_s_raw = model_scratch.predict(frase_clean)
        pred_s = pred_s_raw[0][0].replace('__label__', '').upper()
        conf_s = pred_s_raw[1][0] # La confidenza (da 0 a 1)

        # 3. Predizione con Modello Pre-trained
        pred_p_raw = model_pretrained.predict(frase_clean)
        pred_p = pred_p_raw[0][0].replace('__label__', '').upper()
        conf_p = pred_p_raw[1][0]
        
        # 4. Stampa formattata (tronchiamo la frase se è troppo lunga per la tabella)
        frase_display = (frase[:47] + '..') if len(frase) > 50 else frase
        print(f"{frase_display:<50} | {pred_s} ({conf_s:.2f}) | {pred_p} ({conf_p:.2f})")

    print("="*80)

🚀 Addestramento Modello 1 (SCRATCH)...
🚀 Addestramento Modello 2 (PRE-TRAINED ITALIANO)...
⚠️  Attenzione: Caricamento vettori da 4GB in corso (può richiedere 2-3 min)...

METRICA              | SCRATCH         | PRE-TRAINED (FB)
Precision            | 0.7929          | 0.8143
Recall               | 0.7929          | 0.8143
Tempo Training       | 0.45 sec       | 142.90 sec

FRASE DI TEST                                      | SCRATCH      | PRE-TRAINED
Il personale era abbastanza freddo ma il cibo b..  | NEGATIVE (1.00) | NEGATIVE (1.00)
Un'esperienza indimenticabile, torneremo sicura..  | POSITIVE (1.00) | POSITIVE (1.00)
Pessimo servizio, ho aspettato un'ora per un pi..  | POSITIVE (0.62) | NEGATIVE (0.84)
L'atmosfera è suggestiva e l'arredamento ricerc..  | POSITIVE (1.00) | POSITIVE (1.00)
Locale mediocre, non lo consiglierei ai miei am..  | NEGATIVE (0.91) | NEGATIVE (0.95)
Cibo sublime, ma il conto era davvero esorbitante. | NEGATIVE (1.00) | NEGATIVE (1.00)
Si mangia bene.     